# NeuralMath — A matematikától a Pythonig

Most kizárólag NumPy -val valósítjuk meg a 7 . szakasz előrehaladó számítását, láncszabályát és sztochasztikus frissítéseit [ 4 ] . A nemlineáris döntési határhoz a rejtett réteg neuronjainak számát háromról tizenkettőre növeljük. A \(2\) – \(12\) – \(1\) hálózatnak \(12(2+1)+(12+1)=49\) tanítható paramétere van, de ugyanazok az egyenletek érvényesek.

**NumPy · 2–12–1 · 64 + 8 · 6000 · η = 0.05**

A szintetikus adathalmaz előállítása


## 01 — Környezet előkészítése


In [ ]:
"""Oktatási példa: gyümölcsosztályozás 2-12-1 hálózattal.
Függőségek: numpy és matplotlib. Futtatás: python neural_network.py.
"""
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D




## 02 — ReLU-aktiváció


In [ ]:
def relu(s):
    """A rejtett réteg aktivációja: ReLU(s) = max(0,s)."""
    return np.maximum(0, s)




## 03 — A ReLU számítási deriváltja


In [ ]:
def relu_derivative(s):
    # A ReLU s=0 esetén nem differenciálható; a számítási deriváltat 0-nak választjuk.
    # Ez a konvenció csak a visszaterjesztési lépést definiálja ezen a ponton.
    return (s > 0).astype(float)




## 04 — Szigmoid aktiváció


In [ ]:
def sigmoid(s):
    """Kimeneti aktiváció 0 és 1 közötti értékekkel."""
    return 1 / (1 + np.exp(-s))




## 05 — Segédgörbe az adatok előállításához


In [ ]:
def reference_curve(m):
    # Mesterséges görbe (cm): eltolás, Gauss-domb és logisztikus emelkedés.
    # Az állandókat nemlineáris referenciagörbe létrehozásához választottuk;
    # nem kísérleti mérések és nem illesztett paraméterek.
    return (6.68 + 0.74*np.exp(-((m-155)/20)**2)
            + 1.50/(1 + np.exp(-(m-202)/5)))




## 06 — Szintetikus tanító- és szemléltető tesztadatok


In [ ]:
def construct_data():
    # Mind a négy sávban 16 tömegérték: a1, a2, l1, l2.
    masses_1 = np.linspace(118, 212, 16)
    # Kis determinisztikus eltolások akadályozzák meg az azonos tömegeket.
    masses_2 = masses_1 + np.array([
        1.2, -1.1, 0.8, -1.3, 1.0, -0.8, 1.1, -1.0,
        1.3, -0.9, 0.9, -1.2, 1.0, -1.0, 0.8, -0.7
    ])
    k = np.arange(16)

    # Almák: két sáv a görbe fölött, kis centiméteres eltérésekkel.
    a1 = np.column_stack([
        masses_1,
        reference_curve(masses_1) + 0.22 + 0.05*np.sin(0.8*k)
    ])
    a2 = np.column_stack([
        masses_2,
        reference_curve(masses_2) + 0.58 + 0.06*np.cos(0.45*k)
    ])

    # Narancsok: két sáv a referenciagörbe alatt.
    l1 = np.column_stack([
        masses_1,
        reference_curve(masses_1) - (0.22 + 0.04*np.cos(0.7*k))
    ])
    l2 = np.column_stack([
        masses_2,
        reference_curve(masses_2) - (0.55 + 0.05*np.sin(0.5*k))
    ])

    X_train = np.vstack([a1, a2, l1, l2])
    y_train = np.array([1.0]*32 + [0.0]*32)

    # Nyolc szemléltető példa, nem átfogó statisztikai teszt.
    X_test = np.array([
        [121.0, 7.20], [135.0, 7.45],
        [152.0, 7.75], [171.0, 7.32],
        [121.0, 6.28], [152.0, 6.86],
        [189.0, 6.48], [208.0, 7.55]
    ])
    y_test = np.array([1, 1, 1, 1, 0, 0, 0, 0], dtype=float)
    return X_train, y_train, X_test, y_test




## 07 — Előrehaladó számítás és előrejelzés


In [ ]:
def predict(X, mean, std, W1, b1, W2, b2):
    # x_j^* = (x_j - mu_j)/sigma_j, a tanítóadatok statisztikáival.
    Xn = (X - mean) / std
    # Előrehaladó számítás mátrixjelöléssel (gyümölcsönként egy sor).
    h = relu(Xn @ W1.T + b1)
    return sigmoid(h @ W2 + b2)




## 08 — Tanítás visszaterjesztéssel és gradiensmódszerrel


In [ ]:
def train(Xn, y_train, eta=0.05, n_epochs=6000, seed=42):
    rng = np.random.default_rng(seed)
    # A kis véletlen súlyok megtörik a neuronok szimmetriáját;
    # a kezdeti eltolások nullák. A 0.1 skála oktatási döntés.
    W1 = 0.1 * rng.standard_normal((12, 2))
    b1 = np.zeros(12)
    W2 = 0.1 * rng.standard_normal(12)
    b2 = 0.0
    history = []

    for epoch in range(n_epochs):
        # SGD: gyümölcsönként egy frissítés, korszakonként véletlen sorrendben.
        for i in rng.permutation(len(Xn)):
            x, target = Xn[i], y_train[i]

            # Előrehaladó számítás: a rejtett és kimeneti réteg egyenletei.
            z1 = W1 @ x + b1
            h = relu(z1)
            z2 = W2 @ h + b2
            y_hat = sigmoid(z2)

            # Visszaterjesztés: delta2 = dE/dz2 (négyzetes hiba).
            delta2 = (y_hat - target) * y_hat * (1 - y_hat)
            dW2, db2 = delta2 * h, delta2
            # delta1_j = delta2 * W2_j * ReLU'(z1_j).
            delta1 = (delta2 * W2) * relu_derivative(z1)
            dW1 = np.outer(delta1, x)
            db1 = delta1

            # Sztochasztikus gradienslépés: theta <- theta - eta*dE/dtheta.
            # Minden gradiens a KORÁBBI paraméterértékeket használja.
            W1 -= eta * dW1
            b1 -= eta * db1
            W2 -= eta * dW2
            b2 -= eta * db2

        # J: átlagos tanítási veszteség minden korszak végén.
        h_all = relu(Xn @ W1.T + b1)
        y_pred = sigmoid(h_all @ W2 + b2)
        history.append(np.mean(0.5 * (y_train - y_pred)**2))

    return W1, b1, W2, b2, np.array(history)




## 09 — A megtanult döntési határ ábrázolása


In [ ]:
def plot_results(X_train, y_train, X_test, y_test,
                  mean, std, W1, b1, W2, b2, save_path=None):
    # Az y_hat=0.5 szintvonal a MEGTANULT döntési határ.
    masses = np.linspace(116, 214, 500)
    diameters = np.linspace(6.0, 8.85, 500)
    M, D = np.meshgrid(masses, diameters)
    grid = np.column_stack([M.ravel(), D.ravel()])
    P = predict(grid, mean, std, W1, b1, W2, b2).reshape(M.shape)

    fig, ax = plt.subplots(figsize=(9.2, 6.3))
    ax.contourf(M, D, P, levels=[0, 0.25, 0.50, 0.75, 1],
                colors=['#f3e0cb', '#ecd5c4', '#dbe7cf', '#cde3bf'],
                alpha=0.95)
    ax.contour(M, D, P, levels=[0.5], colors=['#5a1a8a'],
               linewidths=2.2)

    mask = y_train == 1.0
    ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='o',
               s=48, facecolors='white', edgecolors='black', linewidths=1.1)
    mask = y_train == 0.0
    ax.scatter(X_train[mask, 0], X_train[mask, 1], marker='s',
               s=48, facecolors='white', edgecolors='black', linewidths=1.1)
    mask = y_test == 1.0
    ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='^',
               s=92, color='#1f77b4')
    mask = y_test == 0.0
    ax.scatter(X_test[mask, 0], X_test[mask, 1], marker='x',
               s=110, color='#ff7f0e', linewidths=2.0)

    ax.set_xlabel('Tömeg (g)')
    ax.set_ylabel('Átmérő (cm)')
    ax.set_title('A neurális hálózat kimenete tanítás után')
    handles = [
        Line2D([0], [0], marker='o', markersize=9, markerfacecolor='white',
               markeredgecolor='black', linestyle='None',
               label='Alma - tanítás'),
        Line2D([0], [0], marker='s', markersize=9, markerfacecolor='white',
               markeredgecolor='black', linestyle='None',
               label='Narancs - tanítás'),
        Line2D([0], [0], marker='^', markersize=9, color='#1f77b4',
               linestyle='None', label='Alma - teszt'),
        Line2D([0], [0], marker='x', markersize=10, color='#ff7f0e',
               linestyle='None', label='Narancs - teszt'),
        Line2D([0], [0], color='#5a1a8a', linewidth=2.2,
               label='Megtanult döntési határ')
    ]
    ax.legend(handles=handles, loc='center left',
              bbox_to_anchor=(1.02, 0.5), frameon=True)
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=200, bbox_inches='tight')
    return fig, ax




## 10 — A kísérlet futtatása


In [ ]:
if __name__ == '__main__':
    X_train, y_train, X_test, y_test = construct_data()
    mean, std = X_train.mean(axis=0), X_train.std(axis=0)
    Xn = (X_train - mean) / std
    W1, b1, W2, b2, history = train(Xn, y_train)
    p_train = predict(X_train, mean, std, W1, b1, W2, b2)
    p_test = predict(X_test, mean, std, W1, b1, W2, b2)
    print('Végső átlagos veszteség:', history[-1])
    print('Helyes tanítási osztályozások:', np.sum((p_train >= 0.5) == y_train), '/64')
    print('Helyes szemléltető tesztosztályozások:',
          np.sum((p_test >= 0.5) == y_test), '/8')
    plot_results(X_train, y_train, X_test, y_test,
                 mean, std, W1, b1, W2, b2)
    plt.show()
